# Notebook 07 — Concept Drift Detection & Triggered Retraining

**Objective (from your seminar):** "Implement concept drift detection and automated retraining, keeping the model reliable over time."

**What this notebook does, and why:** A model trained once on historical data can silently become inaccurate as legal patterns, procedures, or case-filing behaviour shift — this is called concept drift. This notebook checks for two kinds of drift (has the DATA changed shape? has the MODEL'S ACCURACY dropped?), and only retrains the Cox model if drift is actually detected — never on a fixed schedule — using just the recent data rather than the full history.

---
### ⚠️ Important scoping note
This notebook simulates drift detection using the `period` column (`baseline` vs `monitoring`, split by year in Notebook 03) as a stand-in for "old data" vs "newly arriving data." A real production system would compare against genuinely new cases as they're filed — worth stating clearly as a limitation in your report. Same sandbox limitation as before applies here too — please run and confirm on your machine.

## Step 1 — Import libraries

In [1]:
# pandas / numpy -- for data handling
import pandas as pd
import numpy as np

# CoxPHFitter, concordance_index -- same tools from Notebook 04, since drift
# detection means re-checking (and possibly re-fitting) that same model
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index

# ks_2samp -- the Kolmogorov-Smirnov test, used to check whether two numeric
# distributions (e.g. "duration_days in the old data" vs "duration_days in
# new data") are statistically different from each other
from scipy.stats import ks_2samp

import pickle
import matplotlib.pyplot as plt
from datetime import datetime

print("Libraries imported successfully")


Libraries imported successfully


## Step 2 — Load the dataset and the existing trained Cox model

In [2]:
# Load the structured dataset (has the 'period' column: 'baseline' or
# 'monitoring', assigned by year back in Notebook 03) and the Cox model
# that was trained on it in Notebook 04.
df = pd.read_csv('../data/processed/structured_dataset.csv')

with open('../models/cox_model.pkl', 'rb') as f:
    cph_old = pickle.load(f)

print("Loaded dataset:", df.shape)
print(df['period'].value_counts())


Loaded dataset: (365, 32)
period
monitoring    263
baseline      102
Name: count, dtype: int64


## Step 3 — Split into baseline (old) vs monitoring (recent) periods
This split is what makes drift detection possible at all.

In [3]:
# ============================================================
# WHY split by period at all?
# ============================================================
# Concept drift can only be detected by comparing "what the model learned
# from" against "what's happening now." Since we don't have a live stream of
# brand-new cases arriving, we use the 'period' column (year-based, from
# Notebook 03) as a stand-in: older cases = baseline (what the model was
# effectively trained to expect), recent cases = monitoring (a proxy for
# "newly arriving data" we'd check the model against in a real deployment).
#
# NOTE: this is a simulated drift-check using historical data, not a live
# production monitor -- worth stating clearly in your report as a scoping
# limitation, since a real deployment would compare against genuinely new,
# not-yet-seen cases.

baseline_df = df[df['period'] == 'baseline'].copy()
monitoring_df = df[df['period'] == 'monitoring'].copy()

print(f"Baseline period: {len(baseline_df)} cases")
print(f"Monitoring period: {len(monitoring_df)} cases")


Baseline period: 102 cases
Monitoring period: 263 cases


## Step 4 — Check for FEATURE drift
Has the shape or mix of incoming cases changed? Checked two ways: a statistical test on the numeric duration distribution, and a plain percentage-point comparison of the crime-type mix.

In [4]:
# ============================================================
# CHECK 1: FEATURE DRIFT
# Has the makeup of incoming cases changed shape or category mix?
# ============================================================

# --- Numeric feature: duration_days ---
# ks_2samp compares the two distributions and returns a p-value.
# A LOW p-value (conventionally < 0.05) means the two distributions are
# statistically different -- i.e. likely drift.
ks_stat, ks_pvalue = ks_2samp(baseline_df['duration_days'], monitoring_df['duration_days'])

print("=== Numeric feature drift: duration_days ===")
print(f"KS statistic: {ks_stat:.3f}   p-value: {ks_pvalue:.3f}")
if ks_pvalue < 0.05:
    print("--> Significant drift detected in case-duration distribution.")
else:
    print("--> No significant drift detected in case-duration distribution.")

# --- Categorical feature: crime_type mix ---
# For categories, we simply compare the proportion of each crime type in
# each period -- a large shift (e.g. more than 15 percentage points) in any
# single category is treated as a warning sign.
print("\n=== Categorical feature drift: crime_type mix ===")
baseline_mix = baseline_df['crime_type'].value_counts(normalize=True)
monitoring_mix = monitoring_df['crime_type'].value_counts(normalize=True)
mix_comparison = pd.DataFrame({'baseline_%': baseline_mix * 100, 'monitoring_%': monitoring_mix * 100}).fillna(0)
mix_comparison['change_pct_points'] = mix_comparison['monitoring_%'] - mix_comparison['baseline_%']
print(mix_comparison.round(1))

feature_drift_flag = (ks_pvalue < 0.05) or (mix_comparison['change_pct_points'].abs().max() > 15)
print(f"\nFEATURE DRIFT FLAG: {feature_drift_flag}")


=== Numeric feature drift: duration_days ===
KS statistic: 0.102   p-value: 0.403
--> No significant drift detected in case-duration distribution.

=== Categorical feature drift: crime_type mix ===
                   baseline_%  monitoring_%  change_pct_points
crime_type                                                    
Cyber Crime               9.8          40.3               30.5
Fraud or Cheating        43.1          29.3              -13.9
Theft or Robbery         47.1          30.4              -16.6

FEATURE DRIFT FLAG: True


## Step 5 — Check for PERFORMANCE drift
This is the more important check — even if the data looks similar, has the model's actual prediction accuracy (Concordance Index) dropped on recent cases?

In [6]:
# ============================================================
# CHECK 2: PERFORMANCE DRIFT
# Is the EXISTING model still making good predictions on recent cases?
# ============================================================
# This matters more than feature drift alone -- the data can shift a little
# without hurting predictions, but if the model's actual accuracy (C-index)
# has dropped on recent cases, that's a direct sign it needs attention.

bool_cols = ['landmark_case', 'bail_cancellation_case', 'parity_argument_used', 'has_special_law']
categorical_cols = ['crime_type', 'region_grouped', 'bail_type', 'prior_cases', 'accused_gender']

def encode_for_cox(data, reference_columns):
    data = data.copy()
    for col in bool_cols:
        data[col] = data[col].astype(int)
    encoded = pd.get_dummies(data, columns=categorical_cols, drop_first=False)
    # Align to the model's original training columns (fills missing ones with 0)
    return encoded.reindex(columns=reference_columns, fill_value=0)

# Encode both periods using the OLD model's known training columns
baseline_encoded = encode_for_cox(baseline_df, cph_old.params_.index)
monitoring_encoded = encode_for_cox(monitoring_df, cph_old.params_.index)

# Get the old model's risk scores (predict_partial_hazard) for each group --
# concordance_index needs a risk score, the actual observed duration, and
# the event indicator, to check how well the RANKING of cases matches reality.
baseline_risk = cph_old.predict_partial_hazard(baseline_encoded)
monitoring_risk = cph_old.predict_partial_hazard(monitoring_encoded)

baseline_cindex = concordance_index(baseline_df['duration_days'], -baseline_risk, baseline_df['is_disposed'])
monitoring_cindex = concordance_index(monitoring_df['duration_days'], -monitoring_risk, monitoring_df['is_disposed'])

print(f"C-index on baseline cases:   {baseline_cindex:.3f}")
print(f"C-index on monitoring cases: {monitoring_cindex:.3f}")

cindex_drop = baseline_cindex - monitoring_cindex
print(f"Drop in C-index: {cindex_drop:.3f}")

# A drop of more than 0.05 (a somewhat arbitrary but reasonable starting
# threshold) is treated as meaningful performance degradation.
performance_drift_flag = cindex_drop > 0.05
print(f"\nPERFORMANCE DRIFT FLAG: {performance_drift_flag}")


C-index on baseline cases:   0.625
C-index on monitoring cases: 0.668
Drop in C-index: -0.043

PERFORMANCE DRIFT FLAG: False


## Step 6 — Decide: retrain, or leave the model as-is?
Triggered retraining, not fixed-schedule retraining — this is the key design choice from the literature reviewed in your paper.

In [7]:
# ============================================================
# FINAL DECISION: should we retrain?
# ============================================================
# We only retrain if drift was ACTUALLY detected -- this is "triggered"
# retraining, not retraining on a fixed schedule, matching the approach
# recommended in the drift-detection literature reviewed in our paper.

drift_detected = feature_drift_flag or performance_drift_flag

print(f"Feature drift detected:     {feature_drift_flag}")
print(f"Performance drift detected: {performance_drift_flag}")
print(f"\n{'='*50}")
print(f"OVERALL DECISION: {'RETRAIN MODEL' if drift_detected else 'NO RETRAINING NEEDED'}")
print(f"{'='*50}")


Feature drift detected:     True
Performance drift detected: False

OVERALL DECISION: RETRAIN MODEL


## Step 7 — Retrain on a sliding window (only if drift was detected)
Uses only the recent (monitoring-period) data, not the full history — so old, possibly outdated patterns don't keep diluting what the model learns from new cases.

In [8]:
# ============================================================
# RETRAIN on a SLIDING WINDOW of recent data (only if drift was detected)
# ============================================================
# Rather than retraining on the FULL historical dataset (which would keep
# diluting the new pattern with old, possibly outdated cases forever), we
# retrain using only the monitoring-period data -- a "sliding window" of the
# most recent, most relevant cases. This matches the recommendation from our
# reviewed literature: efficient retraining should use only the relevant
# recent data, not a fixed full-history retrain.

if drift_detected:
    # Re-encode monitoring_df fully (not reindexed to the old model's columns
    # this time -- we WANT the new model to learn from whatever categories
    # actually appear in the recent data, even if that differs slightly from
    # what the old model saw)
    retrain_df = monitoring_df.copy()
    for col in bool_cols:
        retrain_df[col] = retrain_df[col].astype(int)
    retrain_encoded = pd.get_dummies(retrain_df, columns=categorical_cols, drop_first=False)

    model_cols = ['duration_days', 'is_disposed'] + [
        c for c in retrain_encoded.columns if c not in
        ['case_id', 'duration_days', 'is_disposed', 'disposal_days', 'estimation_method',
         'region', 'year', 'period', 'case_text', 'case_text_length']
        and retrain_encoded[c].dtype in [int, float, bool]
    ]
    retrain_final = retrain_encoded[model_cols].copy()
    for col in retrain_final.columns:
        if retrain_final[col].dtype == bool:
            retrain_final[col] = retrain_final[col].astype(int)
    retrain_final = retrain_final.dropna()

    cph_new = CoxPHFitter(penalizer=0.1)  # small penalizer added for stability on a smaller retrain set
    cph_new.fit(retrain_final, duration_col='duration_days', event_col='is_disposed')

    print("Model retrained on monitoring-period data.")
    print(f"Retrained on {len(retrain_final)} cases.")
else:
    cph_new = cph_old
    print("No retraining performed -- no drift was detected, so the existing model is kept as-is.")


Model retrained on monitoring-period data.
Retrained on 263 cases.


## Step 8 — Prove the retraining actually helped
Compares the old model and the new model's Concordance Index on the same recent data, side by side.

In [9]:
# ============================================================
# PROVE the retraining helped: compare OLD vs NEW model on monitoring data
# ============================================================
if drift_detected:
    new_model_encoded = encode_for_cox(monitoring_df, cph_new.params_.index)
    new_model_risk = cph_new.predict_partial_hazard(new_model_encoded)
    new_model_cindex = concordance_index(monitoring_df['duration_days'], -new_model_risk, monitoring_df['is_disposed'])

    print("Concordance Index on monitoring-period cases:")
    print(f"  OLD model (trained on baseline only): {monitoring_cindex:.3f}")
    print(f"  NEW model (retrained on monitoring):  {new_model_cindex:.3f}")

    improvement = new_model_cindex - monitoring_cindex
    print(f"\nImprovement: {improvement:+.3f}")
    if improvement > 0:
        print("--> Retraining improved performance on recent data, as expected.")
    else:
        print("--> Retraining did not improve performance -- worth investigating further "
              "(e.g. too little monitoring data to retrain reliably).")
else:
    print("No comparison needed -- no retraining was triggered.")


Concordance Index on monitoring-period cases:
  OLD model (trained on baseline only): 0.668
  NEW model (retrained on monitoring):  0.665

Improvement: -0.003
--> Retraining did not improve performance -- worth investigating further (e.g. too little monitoring data to retrain reliably).


## Step 9 — Save the (possibly updated) model and a drift-check log
The log keeps a running history of every drift check — useful evidence for your report/viva, showing this isn't a one-off check but a repeatable monitoring process.

In [10]:
# ============================================================
# Save the (possibly updated) model, plus a drift-check LOG entry
# ============================================================
# Keeping a log is important for accountability -- your report/viva can show
# a history of when drift was checked, what was found, and what action was
# taken, rather than just the final state.

log_entry = {
    'check_timestamp': datetime.now().isoformat(),
    'baseline_cases': len(baseline_df),
    'monitoring_cases': len(monitoring_df),
    'feature_drift_detected': bool(feature_drift_flag),
    'performance_drift_detected': bool(performance_drift_flag),
    'baseline_cindex': round(float(baseline_cindex), 3),
    'monitoring_cindex_old_model': round(float(monitoring_cindex), 3),
    'retrained': bool(drift_detected),
}

log_df = pd.DataFrame([log_entry])

# Append to an existing log file if one exists, otherwise create it --
# this way every time this notebook runs, it adds a new row to the history.
import os
log_path = '../data/processed/drift_log.csv'
if os.path.exists(log_path):
    existing_log = pd.read_csv(log_path)
    log_df = pd.concat([existing_log, log_df], ignore_index=True)
log_df.to_csv(log_path, index=False)

if drift_detected:
    with open('../models/cox_model.pkl', 'wb') as f:
        pickle.dump(cph_new, f)
    print("Updated model saved -> models/cox_model.pkl (overwriting the old one)")

print("Drift-check log saved/updated -> data/processed/drift_log.csv")
print(log_df.tail(1))


Updated model saved -> models/cox_model.pkl (overwriting the old one)
Drift-check log saved/updated -> data/processed/drift_log.csv
              check_timestamp  baseline_cases  monitoring_cases  \
0  2026-09-22T01:44:18.884979             102               263   

   feature_drift_detected  performance_drift_detected  baseline_cindex  \
0                    True                       False            0.625   

   monitoring_cindex_old_model  retrained  
0                        0.668       True  
